In [10]:
# imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import sys
import os

sys.path.append(os.path.abspath('..'))

from read_db import loadAlbumsDf

In [ ]:
# load dataframe
df = loadAlbumsDf("/media/shared/projects/albumify/src/backend/app/db/albumify.db")

In [6]:
df.tail()

,album_id,album_name,release_date,album_popularity,avg_track_duration,artist_names,avg_artist_popularity,genres,album_tags,artist_tags
691,7teTxSPnJrpRoi7BxJ9qTC,American Nights - In Concert,1991,41,272219.483871,[The Doors],72.0,"[classic rock, psychedelic rock, acid rock]",None,"[{'tag': 'rock', 'weight': 47}, {'tag': 'class..."
692,7yQtjAjhtNi76KRu05XWFS,Grace,1994,78,311702.727273,[Jeff Buckley],74.0,None,"[{'tag': 'alternative', 'weight': 7}, {'tag': ...","[{'tag': 'indie', 'weight': 15}, {'tag': 'alte..."
693,7ycBtnsMtyVbbwTfJwRjSP,To Pimp A Butterfly,2015-03-16,75,296225.562500,[Kendrick Lamar],89.0,"[hip hop, west coast hip hop]","[{'tag': 'hip-hop', 'weight': 4}, {'tag': 'fun...","[{'tag': 'hip-hop', 'weight': 100}, {'tag': 'r..."
694,7ykAHaoptbCYaO0HAjpgcL,The Cry of Love,1971,43,240086.300000,[Jimi Hendrix],67.0,"[classic rock, psychedelic rock, acid rock, bl...","[{'tag': 'rock', 'weight': 44}, {'tag': 'hard ...","[{'tag': 'rock', 'weight': 75}, {'tag': 'hard ..."
695,7zL4It7y3FlhXduscGWo9a,Going Back To Colorado / Leaving Colorado,2016-07-08,3,410723.800000,[Zephyr],18.0,[blues rock],None,"[{'tag': 'rock', 'weight': 19}, {'tag': 'femal..."


In [4]:
df.isna().sum()

album_id                   0
album_name                 0
release_date               0
album_popularity           0
avg_track_duration         0
artist_names               0
avg_artist_popularity      0
genres                   114
album_tags                89
artist_tags               10
dtype: int64

In [7]:
df.dtypes

album_id                     str
album_name                   str
release_date              object
album_popularity           int64
avg_track_duration       float64
artist_names              object
avg_artist_popularity    float64
genres                    object
album_tags                object
artist_tags               object
dtype: object

In [8]:
# first convert release date to release years for consistency
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
df['release_year'] = df['release_year'].fillna(df['release_year'].median()).astype(int)
df.drop(columns=['release_date'], inplace=True)

In [14]:
# then multi-hot encode genres
mlbGenres = MultiLabelBinarizer()
genreMatrix = mlbGenres.fit_transform(df['genres'].apply(lambda x: x if x else []))
genreDf = pd.DataFrame(genreMatrix, columns=[f"genre_{g}" for g in mlbGenres.classes_])

In [16]:
# then multi-hot encode tags 
all_tags = set(t['tag'] for tags in df['album_tags'].dropna() for t in tags)

def buildTagVector(tags, all_tags):
    vec = {f"atag_{t}": 0 for t in all_tags}
    if tags:
        for t in tags:
            vec[f"atag_{t['tag']}"] = t['weight']
    return vec

tag_df = pd.DataFrame(df['album_tags'].apply(lambda x: buildTagVector(x, all_tags)).tolist())

In [17]:
tag_df

,atag_jazz hop,atag_atlanta,atag_bay area,atag_classic track,atag_christmas pop,atag_russia,atag_lullabies,atag_covers,atag_clube da esquina,atag_abstract hip hop,...,atag_santana,atag_british hip hop,atag_hip house,atag_jbtv recommendation,atag_cadderley,atag_karlsruhe,atag_supertramp,atag_8 out of 10,atag_deja vu,atag_::get::
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
691,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
692,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
693,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
694,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
